# Notebook 2 — Pixel PPO: Train on World 1-1

Trains a **PPO** agent on World 1-1 using raw pixel observations (84×84 grayscale). This is the pixel-based baseline — the CNN policy processes visual frames, unlike the symbolic pipeline used in all other notebooks.

## Study Progression
1. Symbolic DQN — World 1-1
2. **→ Pixel PPO — World 1-1 (you are here)**
3. Symbolic PPO — World 1-1
4. Transfer Learning — Symbolic PPO 1-1 → 1-2
5. Multi-Task PPO — World 1-1 + 1-2
6. World 1 Full — Random Starts + All 4 Levels

## Setup
- **Observation:** 84×84 grayscale × 4 frame-stack → `(84, 84, 4)`
- **Policy:** `CnnPolicy` (Nature CNN)
- **Training:** 8 parallel envs, two phases (2M + 2M steps)
- **Output:** `models/pixel_ppo_w1l1/`

In [ ]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    !pip install tensordict torchrl
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_pixel_vec_env, make_pixel_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Create 8 parallel pixel environments for World 1-1
NUM_ENVS = 8

env = make_pixel_vec_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4,
    num_envs=NUM_ENVS,
)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')
print(f'Parallel envs: {NUM_ENVS}')

In [ ]:
# Phase 1: Train from scratch — 2M steps
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LOG_DIR = 'logs/pixel_ppo_w1l1'
SAVE_DIR = 'models/pixel_ppo_w1l1'

config = PPOConfig(
    policy='CnnPolicy',
    learning_rate=2.5e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.9,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    total_timesteps=2_000_000,
)

model = PPO(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    n_steps=config.n_steps,
    batch_size=config.batch_size,
    n_epochs=config.n_epochs,
    gamma=config.gamma,
    gae_lambda=config.gae_lambda,
    clip_range=config.clip_range,
    ent_coef=config.ent_coef,
    vf_coef=config.vf_coef,
    max_grad_norm=config.max_grad_norm,
    tensorboard_log=LOG_DIR,
    verbose=1,
    device=DEVICE,
)

print(f'Phase 1: {config.total_timesteps:,} timesteps')
print(f'Device: {model.device}')
print(f'Batch per rollout: {config.n_steps * NUM_ENVS} samples')

In [ ]:
# Launch TensorBoard (Colab inline)
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

In [ ]:
# Phase 1: Train for 2M steps
callback = CheckpointAndLogCallback(
    save_path=SAVE_DIR,
    save_freq=50_000,
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=1,
)
model.save(f'{SAVE_DIR}/phase1_model')
print('Phase 1 complete!')

In [ ]:
# Phase 2: Fine-tune with lower LR — 2M more steps
model2 = PPO.load(f'{SAVE_DIR}/phase1_model', env=env, device=DEVICE)
model2.learning_rate = 1e-5

callback2 = CheckpointAndLogCallback(
    save_path=SAVE_DIR,
    save_freq=50_000,
)

model2.learn(
    total_timesteps=2_000_000,
    callback=callback2,
    log_interval=1,
)
model2.save(f'{SAVE_DIR}/final_model')
print('Phase 2 complete — final_model saved!')

In [ ]:
# Plot training curves (both phases combined)
import matplotlib.pyplot as plt
import numpy as np

all_rewards = callback.episode_rewards + callback2.episode_rewards
all_lengths = callback.episode_lengths + callback2.episode_lengths
all_flags   = callback.episode_flags   + callback2.episode_flags

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
window = min(100, len(all_rewards))

for ax, data, color, title in zip(
    axes,
    [all_rewards, all_lengths, None],
    ['blue', 'orange', 'green'],
    ['Episode Rewards', 'Episode Lengths', 'Cumulative Flag Rate'],
):
    if title == 'Cumulative Flag Rate':
        if all_flags:
            cum = np.cumsum(all_flags) / (np.arange(len(all_flags)) + 1)
            ax.plot(cum, color=color, linewidth=2)
            ax.set_ylim(0, 1)
    else:
        ax.plot(data, alpha=0.3, color=color)
        if len(data) > window:
            smoothed = np.convolve(data, np.ones(window) / window, mode='valid')
            ax.plot(range(window - 1, len(data)), smoothed, color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Episode')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Retrieve TensorBoard metrics programmatically
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

# Find the latest TensorBoard event file
event_files = sorted(glob.glob(f'{LOG_DIR}/**/events.out.tfevents.*', recursive=True))
print(f'Found {len(event_files)} TensorBoard event file(s)')

if event_files:
    ea = EventAccumulator(os.path.dirname(event_files[-1]))
    ea.Reload()

    print('\nAvailable scalar tags:')
    for tag in ea.Tags()['scalars']:
        print(f'  - {tag}')

    # Extract key metrics
    metrics = {}
    for tag in ea.Tags()['scalars']:
        events = ea.Scalars(tag)
        metrics[tag] = {
            'steps': [e.step for e in events],
            'values': [e.value for e in events],
        }

    # Plot key SB3 metrics from TensorBoard
    tb_keys = [
        'rollout/ep_rew_mean',
        'rollout/ep_len_mean',
        'train/loss',
        'train/entropy_loss',
        'train/policy_gradient_loss',
        'train/value_loss',
    ]
    available_keys = [k for k in tb_keys if k in metrics]

    if available_keys:
        n_plots = len(available_keys)
        cols = min(3, n_plots)
        rows = (n_plots + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
        if n_plots == 1:
            axes = [axes]
        else:
            axes = axes.flatten()

        for ax, key in zip(axes, available_keys):
            ax.plot(metrics[key]['steps'], metrics[key]['values'], linewidth=1.5)
            ax.set_title(key)
            ax.set_xlabel('Timesteps')
            ax.grid(True, alpha=0.3)

        # Hide unused axes
        for ax in axes[len(available_keys):]:
            ax.set_visible(False)

        plt.tight_layout()
        plt.savefig(f'{SAVE_DIR}/tensorboard_metrics.png', dpi=150)
        plt.show()
    else:
        print('No matching scalar tags found.')

In [ ]:
# Evaluate final model — 10 episodes
import numpy as np
from stable_baselines3 import PPO

eval_model = PPO.load(f'{SAVE_DIR}/final_model')

eval_env = make_pixel_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4,
)

rewards, lengths, flags = [], [], []

for ep in range(10):
    reset_result = eval_env.reset()
    obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    done, total_reward, steps, flag = False, 0.0, 0, False

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        result = eval_env.step(int(action))
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            obs, reward, done, info = result
        total_reward += float(reward)
        steps += 1
        if isinstance(info, dict) and info.get('flag_get', False):
            flag = True

    rewards.append(total_reward)
    lengths.append(steps)
    flags.append(flag)
    print(f'Episode {ep+1:2d}: reward={total_reward:.1f}, steps={steps}, {"FLAG!" if flag else "DEAD"}')

print(f'\nMean reward: {np.mean(rewards):.1f} ± {np.std(rewards):.1f}')
print(f'Mean length: {np.mean(lengths):.0f}')
print(f'Flag rate:   {np.mean(flags):.0%}')

eval_env.close()

In [ ]:
# Download model from Colab (optional)
if os.getenv('COLAB_RELEASE_TAG'):
    from google.colab import files
    import shutil
    shutil.make_archive('pixel_ppo_w1l1', 'zip', SAVE_DIR)
    files.download('pixel_ppo_w1l1.zip')
    print('Model archive downloaded!')